In [1]:
import pandas as pd
from loguru import logger
import os

In [2]:
COLS_TO_DROP = [
    # --- Negligible numerical ---
    "mths_since_last_delinq",
    "mths_since_last_record",
    "total_acc",
    "collections_12_mths_ex_med",
    "mths_since_last_major_derog",
    "acc_now_delinq",
    "tot_coll_amt",
    "mths_since_rcnt_il",
    "total_bal_il",
    "max_bal_bc",
    "total_cu_tl",
    "mo_sin_old_il_acct",
    "num_il_tl",
    "pct_tl_nvr_dlq",
    # --- Unanalysed / low-info numerical ---
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl",
    "mths_since_recent_bc",
    "mths_since_recent_inq",
    "mths_since_recent_revol_delinq",
    "num_actv_bc_tl",
    "num_bc_sats",
    "num_bc_tl",
    "num_sats",
    "num_tl_120dpd_2m",
    "num_tl_30dpd",
    "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m",
    "tax_liens",
    "total_bal_ex_mort",
    "total_il_high_credit_limit",
    "bc_open_to_buy",
    # --- Joint application columns ---
    "application_type",           # after filtering Individual
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog",
    # --- Categorical drops ---
    "emp_title",                  # 300k+ unique values
]

In [3]:
file_path = '../data/raw/loan_preprocessed.parquet'

def read_data(file_path = file_path):
    """Reads the dataset from the preprocessed data - Parquet file."""
    logger.info(f"Reading data from {file_path}...")
    try: 
        df = pd.read_parquet(file_path)
        logger.success("Data read successfully.")
        return df
    except Exception as e:
        logger.error(f"Error reading data: {e}")
        raise e 


df = read_data()
df.sample(10)
# rename target column for use of helper functions




2026-03-11 02:07:03.681 | INFO     | __main__:read_data:5 - Reading data from ../data/raw/loan_preprocessed.parquet...
2026-03-11 02:07:04.296 | SUCCESS  | __main__:read_data:8 - Data read successfully.


,loan_amnt,term,emp_title,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,...,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog
443158,15000,36 months,nurse,6 years,MORTGAGE,33000.0,Verified,Fully Paid,credit_card,OH,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1054751,10000,36 months,laboratory technician,5 years,OWN,65000.0,Source Verified,Charged Off,home_improvement,MD,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1146037,12000,36 months,Manager,2 years,RENT,70000.0,Verified,Fully Paid,credit_card,NY,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1066682,6000,36 months,Loos prevention,2 years,RENT,29000.0,Not Verified,Fully Paid,credit_card,FL,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
906510,16000,36 months,Gentiva Health Services,2 years,MORTGAGE,65000.0,Verified,Fully Paid,credit_card,GA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
851058,8725,36 months,rda,< 1 year,RENT,34000.0,Verified,Charged Off,debt_consolidation,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
769743,20000,60 months,technician,10+ years,MORTGAGE,51030.0,Verified,Fully Paid,house,AZ,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1238420,25000,60 months,Granite Construction Inc,5 years,MORTGAGE,175000.0,Source Verified,Fully Paid,debt_consolidation,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200394,8500,36 months,Software,3 years,RENT,47000.0,Source Verified,Charged Off,debt_consolidation,OR,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
369233,12625,36 months,Bus operator,10+ years,MORTGAGE,80000.0,Not Verified,Fully Paid,debt_consolidation,MN,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

df_new= df.drop(columns=COLS_TO_DROP, errors='ignore')

df_new.rename(columns={'loan_status': 'target'}, inplace=True)

df_new['target'] = df_new['target'].map({
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1    

})

# Step 1 — drop NONE rows (48 rows, statistically unreliable default rate)
none_count = (df_new["home_ownership"] == "NONE").sum()
if none_count > 0:
    df_new.drop(index=df_new[df_new["home_ownership"] == "NONE"].index, inplace=True)
    logger.warning(f"  Dropped {none_count} rows where home_ownership == 'NONE'")

2026-03-11 02:07:47.158 | WARNING  | __main__:<module>:16 -   Dropped 48 rows where home_ownership == 'NONE'


In [5]:
df_new.target

0          0
1          0
2          0
3          0
4          0
          ..
1303633    1
1303634    1
1303635    0
1303636    0
1303637    0
Name: target, Length: 1303590, dtype: int64

In [6]:
df_new.columns

Index(['loan_amnt', 'term', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'target', 'purpose', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'il_util', 'open_rv_12m',
       'open_rv_24m', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_util',
       'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_rev_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'percent_bc_gt_75',
       'pub_rec_bankruptcies', 'tot_hi_cred_lim', 'total_bc_limit'],
      dtype='object')

In [7]:
OUTPUT_DIR = "../data/processed"
df_new.to_parquet(f"{OUTPUT_DIR}/loan_selected.parquet", index=False)
